# Skill 43: Semantic Chunking Algorithm

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Split a long document into sentence-aware, overlapping chunks sized for embedding and retrieval.

## Objectives
- Preserve sentence boundaries when possible.
- Add overlap so neighbouring chunks retain context.
- Handle oversized sentences and avoid infinite loops.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
Stride $S=W-O$; chunks $\approx\lceil (N-O)/(W-O)\rceil$; redundancy $W/(W-O)$ (e.g. N=1000, W=80, O=15 gives 16 chunks, 1.23x).

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import re

def semantic_chunks(text, max_words=80, overlap_words=15):
    if max_words <= 0 or overlap_words < 0 or overlap_words >= max_words:
        raise ValueError("Require max_words > overlap_words >= 0")
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current = [], []

    def flush():
        nonlocal current
        if current:
            chunks.append(" ".join(current).strip())
            words = " ".join(current).split()
            current = words[-overlap_words:] if overlap_words else []

    for sentence in sentences:
        if not sentence:
            continue
        words = sentence.split()
        # Break a sentence that alone exceeds the target.
        while len(words) > max_words:
            if current:
                flush()
            chunks.append(" ".join(words[:max_words]))
            words = words[max_words - overlap_words:] if overlap_words else words[max_words:]
        if len(" ".join(current + words).split()) > max_words:
            flush()
        current.extend(words)
    if current:
        chunks.append(" ".join(current).strip())
    return [c for c in chunks if c]

text = ("Retrieval augmented generation grounds answers in external documents. "
        "Chunking determines the context units stored in a vector database. "
        "Overlap can preserve facts split across sentence boundaries. " * 8)
for i, chunk in enumerate(semantic_chunks(text, max_words=25, overlap_words=5), 1):
    print(f"CHUNK {i} ({len(chunk.split())} words): {chunk}\n")


## What to check
**Expected result:** Chunks stay within the word limit and each starts with the last 5 words of the previous one.

## Pitfalls
Known limitation: a long sentence after a short one can emit the saved overlap out of order. Whitespace words are only a proxy for tokens; abbreviations break the sentence regex.